# Event-Triggered Dual Heuristic Programming with in-flight damage on the nonlinear F-16

This notebook demonstrates the **ET-DHP** agent adapting online to a real damage event during sinusoidal angle-of-attack tracking on the pure-NumPy nonlinear F-16 longitudinal model.

**Scenario:**

* Total simulation time: **60 s** (training + final eval).
* Damage trigger: **t = 20 s**.
* Damage type: symmetric 30 % loss of both wing tips, applied through the proper damage subsystem (`tensoraerospace.aerospacemodel.f16.nonlinear.damage`).
* The env recomputes effective wing area `S`, MAC, and inertias via Huygens-Steiner; the longitudinal lift coefficient drops via strip-theory corrections.
* Online ET-DHP learning stays active during the damaged eval — the plant NN is frozen (it was pre-trained on the healthy aircraft), but the actor and critic re-train when the Lipschitz event trigger fires.

**Reference:** Bo Sun, Cheng Liu, Killian Dally, Erik-Jan van Kampen. *"Intelligent Aircraft Stabilization Control with Event-Triggered Scheme"*, CEAS EuroGNC 2022.

**See also:** `example_iadp_damage_f16.py` — same scenario with the iADP agent, which adapts more aggressively because its RLS identifier picks up the post-damage plant gain in real time.

## 1. Imports and simulation settings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import fsolve

import tensoraerospace.envs  # noqa: F401 — registers gym envs
from tensoraerospace.aerospacemodel.f16.nonlinear.damage import (
    DamageEvent,
    DamageProfile,
)
from tensoraerospace.aerospacemodel.f16.nonlinear.longitudinal.dynamics import f16_ode_long
from tensoraerospace.aerospacemodel.f16.nonlinear.longitudinal.params import default_parameters
from tensoraerospace.agent.et_dhp import ETDHPAgent, ETDHPConfig
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

# Simulation settings
DT = 0.01
TOTAL_TIME = 60.0
DAMAGE_TIME = 20.0
DAMAGE_STEP = int(DAMAGE_TIME / DT)

# Reference signal settings
AMPLITUDE_DEG = 3.0
FREQ_HZ = 0.1
WARMUP_SEC = 2.0

# Feedforward settings
LOOKAHEAD_SEC = 0.85
FF_GAIN = 1.55

# Training settings
NUM_TRAIN_EPISODES = 6

print(f'Total simulation time: {TOTAL_TIME:.0f} s')
print(f'Damage trigger:        t = {DAMAGE_TIME:.0f} s (step {DAMAGE_STEP})')

## 2. Global trim and feedforward elevator curve

Solve for `(α*, δₑ*)` such that `dα/dt = dwz/dt = 0`. Then build the inverse-model feedforward curve `δₑ_trim(α)` that zeroes the pitching moment at any commanded angle of attack. The agent learns a residual on top of this curve, so even a freshly-initialised actor tracks reasonably well.

In [ ]:
params = default_parameters()

def trim_residual(z):
    alpha, stab = z
    x = np.array([alpha, 0.0, stab, 0.0])
    return list(f16_ode_long(x, np.array([stab]), 0.0, params)[:2])

sol, _info, ier, msg = fsolve(
    trim_residual, x0=[math.radians(2.0), math.radians(-2.0)], full_output=True,
)
assert ier == 1, f'trim search failed: {msg}'
alpha_trim_rad, stab_trim_rad = float(sol[0]), float(sol[1])
alpha_trim_deg = math.degrees(alpha_trim_rad)
stab_trim_deg = math.degrees(stab_trim_rad)
print(f'global trim:  α* = {alpha_trim_deg:+.4f}°,  δₑ* = {stab_trim_deg:+.4f}°')

In [ ]:
def stab_for_alpha(alpha_rad: float) -> float:
    """Elevator deflection (rad) producing zero pitching moment at given alpha."""
    def res(s):
        x = np.array([alpha_rad, 0.0, s[0], 0.0])
        return f16_ode_long(x, np.array([s[0]]), 0.0, params)[1]
    return float(fsolve(res, x0=[math.radians(-2.0)])[0])

alphas_grid_deg = np.linspace(-12.0, 17.0, 61)
alphas_grid_rad = np.deg2rad(alphas_grid_deg)
stabs_grid_deg = np.array([math.degrees(stab_for_alpha(a)) for a in alphas_grid_rad])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas_grid_deg, stabs_grid_deg, label=r'$\delta_e^{\mathrm{trim}}(\alpha)$')
ax.axvline(alpha_trim_deg, color='tab:gray', linestyle=':', label=r'global $\alpha_{\mathrm{trim}}$')
ax.axhline(stab_trim_deg, color='tab:gray', linestyle=':')
ax.set_xlabel(r'$\alpha$, deg')
ax.set_ylabel(r'$\delta_e^{\mathrm{trim}}$, deg')
ax.set_title('Feedforward trim curve')
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Reference signal (60 s, sinusoidal α around trim)

In [ ]:
tp = generate_time_period(tn=int(TOTAL_TIME), dt=DT)
tps = convert_tp_to_sec_tp(tp, dt=DT)
number_time_steps = len(tp)
warmup_steps = int(WARMUP_SEC / DT)

ref_alpha_rad = np.full(number_time_steps, alpha_trim_rad)
active_t = np.arange(number_time_steps - warmup_steps) * DT
ref_alpha_rad[warmup_steps:] = (
    alpha_trim_rad + math.radians(AMPLITUDE_DEG) * np.sin(2 * np.pi * FREQ_HZ * active_t)
)
reference_signals = ref_alpha_rad.reshape(1, -1)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tps, np.rad2deg(reference_signals[0]), label=r'$\alpha_{\mathrm{ref}}$')
ax.axhline(alpha_trim_deg, color='tab:gray', linestyle=':', label='trim')
ax.axvline(DAMAGE_TIME, color='red', linestyle='--', alpha=0.4, label=f'damage @ t={DAMAGE_TIME:.0f}s')
ax.set_xlabel('time, s')
ax.set_ylabel(r'$\alpha_{\mathrm{ref}}$, deg')
ax.set_title('Sinusoidal angle-of-attack reference (centered on trim)')
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

## 4. State transform, feedforward function, env builder

The env adds the inverse-model feedforward elevator to the agent's residual action before stepping the plant. The agent observes a re-centered state where `α - α_ref` and `δ_stab - δ_trim(α_ref)` are zero at perfect tracking, so the regulator's natural fixed point coincides with reference tracking.

In [ ]:
lookahead_steps = int(LOOKAHEAD_SEC / DT)

def feedforward_fn(time_step: int, ref_signal: np.ndarray) -> float:
    """Inverse-model feedforward with temporal lookahead and amplitude gain."""
    k = min(time_step + lookahead_steps, ref_signal.shape[1] - 1)
    alpha_ref_deg = math.degrees(ref_signal[0, k])
    alpha_eff_deg = alpha_trim_deg + FF_GAIN * (alpha_ref_deg - alpha_trim_deg)
    return float(np.interp(alpha_eff_deg, alphas_grid_deg, stabs_grid_deg))


def state_transform(obs: np.ndarray, ref, ts: int) -> np.ndarray:
    """Convert raw [alpha, wz, stab, dstab] (rad) obs into deg regulation state."""
    obs_deg = np.degrees(np.asarray(obs, dtype=np.float64).reshape(-1))
    if ref is not None and int(ts) < ref.shape[1]:
        alpha_ref_deg_now = math.degrees(float(ref[0, int(ts)]))
    else:
        alpha_ref_deg_now = alpha_trim_deg
    stab_ref_deg_now = float(np.interp(alpha_ref_deg_now, alphas_grid_deg, stabs_grid_deg))
    return np.array([
        obs_deg[0] - alpha_ref_deg_now,
        obs_deg[1],
        obs_deg[2] - stab_ref_deg_now,
        obs_deg[3],
    ])


def make_env(damage_profile: DamageProfile | None = None):
    """Build the tracking env with feedforward active."""
    return gym.make(
        'NonlinearLongitudinalF16-v0',
        number_time_steps=number_time_steps,
        initial_state=[alpha_trim_rad, 0.0, stab_trim_rad, 0.0],
        reference_signal=reference_signals,
        state_space=['alpha', 'wz', 'stab', 'dstab'],
        control_space=['stab'],
        tracking_states=['alpha'],
        use_reward=False,
        dt=DT,
        integrator='euler',
        feedforward_fn=feedforward_fn,
        damage_profile=damage_profile,
    ).unwrapped

## 5. Plant-model offline pre-training (PE around trim)

30 s of multi-sine excitation around the global trim, with the feedforward off (constant trim bias) so the PE data covers the full elevator-to-α chain without reference-dependent offset. The plant NN is trained on this data once and then frozen — it does **not** see the damaged dynamics.

In [ ]:
N_ID = 3000
U_PE_AMP = 2.0

env_id = gym.make(
    'NonlinearLongitudinalF16-v0',
    number_time_steps=N_ID + 2,
    initial_state=[alpha_trim_rad, 0.0, stab_trim_rad, 0.0],
    reference_signal=np.full((1, N_ID + 2), alpha_trim_rad),
    state_space=['alpha', 'wz', 'stab', 'dstab'],
    control_space=['stab'],
    tracking_states=['alpha'],
    use_reward=False,
    dt=DT,
    integrator='euler',
    control_bias=stab_trim_deg,
).unwrapped

rng = np.random.default_rng(0)
states_buf, actions_buf, next_states_buf = [], [], []
ref_at_trim = np.full((1, 1), alpha_trim_rad)
obs, _ = env_id.reset()
for t in range(N_ID):
    u_t = U_PE_AMP * (
        0.6 * np.sin(2 * np.pi * 0.3 * t * DT)
        + 0.3 * np.sin(2 * np.pi * 0.9 * t * DT)
        + 0.1 * rng.normal()
    )
    x_curr = state_transform(obs, ref_at_trim, 0)
    obs_next, _, done, _, _ = env_id.step(np.array([u_t]))
    x_next = state_transform(obs_next, ref_at_trim, 0)
    states_buf.append(x_curr)
    actions_buf.append([u_t])
    next_states_buf.append(x_next)
    obs = obs_next
    if done:
        break

states_arr = np.asarray(states_buf, dtype=np.float32)
actions_arr = np.asarray(actions_buf, dtype=np.float32)
next_states_arr = np.asarray(next_states_buf, dtype=np.float32)
print(f'collected {states_arr.shape[0]} transitions')

In [ ]:
cfg = ETDHPConfig(
    actor_hidden=(24, 24),
    critic_hidden=(24, 24),
    model_hidden=(24, 24),
    actor_lr=1e-3,
    critic_lr=1e-3,
    model_lr=5e-3,
    model_epochs=300,
    Q=[10.0, 0.1, 0.0, 0.0],
    R=[1.0],
    gamma=0.95,
    num_epochs_per_trigger=3,
    u_bound=2.0,
    rho=0.2,
    trigger_floor=0.1,
    weight_init_scale=0.2,
    seed=0,
)

agent = ETDHPAgent(
    n_state=4,
    n_control=1,
    state_transform=state_transform,
    config=cfg,
)

model_losses = agent.fit_plant_model(
    states_arr, actions_arr, next_states_arr,
    batch_size=128, verbose=False,
)

plt.figure(figsize=(7, 3))
plt.semilogy(model_losses)
plt.xlabel('epoch'); plt.ylabel('plant-model MSE'); plt.grid(alpha=0.3)
plt.title('Plant-model offline pre-training (healthy aircraft)')
plt.tight_layout(); plt.show()
print(f'final MSE: {model_losses[-1]:.3e}')

## 6. Closed-loop training on the healthy aircraft

Run `NUM_TRAIN_EPISODES` episodes, each 60 s long, with online actor/critic updates gated by the Lipschitz event trigger. The plant network stays frozen at its offline-pre-trained weights. Training curves typically show RMSE collapsing within 3-4 episodes from a near-FF baseline to ~0.1° late-window.

In [ ]:
def run_episode(agent, env_factory, *, learn: bool):
    """Run a 60 s episode. Returns logs and trigger statistics split by
    pre/post damage-time window."""
    env = env_factory()
    obs, _ = env.reset()
    agent.reset()
    alpha_log, wz_log, u_log = [], [], []
    triggers_pre = triggers_post = 0
    triggered_events: list[tuple[float, str]] = []
    n_steps = number_time_steps - 2
    for k in range(n_steps):
        agent.predict(obs, reference_signals, k)
        u_cmd = agent.last_action()
        obs_next, _, done, _, info = env.step(u_cmd)
        if learn:
            metrics = agent.learn(obs_next, reference_signals, k, dt=DT)
            if metrics['triggered']:
                if k < DAMAGE_STEP:
                    triggers_pre += 1
                else:
                    triggers_post += 1
        for label in info.get('damage_events_triggered', []) or []:
            triggered_events.append((k * DT, label))
        obs_arr = np.asarray(obs_next).reshape(-1)
        alpha_log.append(np.degrees(obs_arr[0]))
        wz_log.append(np.degrees(obs_arr[1]))
        u_log.append(float(u_cmd[0]))
        obs = obs_next
        if done:
            break
    return {
        'alpha': np.asarray(alpha_log),
        'wz': np.asarray(wz_log),
        'u': np.asarray(u_log),
        'triggers_pre': triggers_pre,
        'triggers_post': triggers_post,
        'triggered_events': triggered_events,
    }


def healthy_env():
    return make_env(damage_profile=None)

In [ ]:
train_curve = []
for ep in range(NUM_TRAIN_EPISODES):
    log = run_episode(agent, healthy_env, learn=True)
    n_alpha = len(log['alpha'])
    ref_deg = np.rad2deg(reference_signals[0, :n_alpha])
    late = np.arange(n_alpha // 2, n_alpha)
    rmse = float(np.sqrt(np.mean((log['alpha'][late] - ref_deg[late]) ** 2)))
    n_trig = log['triggers_pre'] + log['triggers_post']
    train_curve.append((ep + 1, rmse, n_trig))
    print(f'  ep {ep + 1}/{NUM_TRAIN_EPISODES}: RMSE_late={rmse:.4f}°  triggers={n_trig}')

ep_idx = [r[0] for r in train_curve]
rmse_vals = [r[1] for r in train_curve]
n_trigs = [r[2] for r in train_curve]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(ep_idx, rmse_vals, '-o')
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('episode'); axes[0].set_ylabel('late-half RMSE, deg')
axes[0].set_title('Tracking error during training')
axes[1].plot(ep_idx, n_trigs, '-o', color='tab:green')
axes[1].set_xlabel('episode'); axes[1].set_ylabel('triggers per episode')
axes[1].grid(alpha=0.3); axes[1].set_title('Event-trigger firings per episode')
plt.tight_layout(); plt.show()

## 7. Final evaluation — baseline (no damage)

Run one more 60 s episode with online learning still enabled, on the **healthy** aircraft. This sets the apples-to-apples baseline against the damaged eval below — the agent in both cases keeps adapting through the event trigger.

In [ ]:
baseline = run_episode(agent, healthy_env, learn=True)
n_alpha = len(baseline['alpha'])
ref_deg = np.rad2deg(reference_signals[0, :n_alpha])
pre = np.arange(int(5.0 / DT), DAMAGE_STEP)
post = np.arange(DAMAGE_STEP + int(2.0 / DT), n_alpha)

pre_rmse_b = float(np.sqrt(np.mean((baseline['alpha'][pre] - ref_deg[pre]) ** 2)))
pre_mae_b  = float(np.mean(np.abs(baseline['alpha'][pre] - ref_deg[pre])))
post_rmse_b = float(np.sqrt(np.mean((baseline['alpha'][post] - ref_deg[post]) ** 2)))
post_mae_b  = float(np.mean(np.abs(baseline['alpha'][post] - ref_deg[post])))

print('=== Baseline (no damage) ===')
print(f'Pre-damage  (5–20 s):    MAE={pre_mae_b:.4f}°  RMSE={pre_rmse_b:.4f}°')
print(f'Post-damage (22–60 s):   MAE={post_mae_b:.4f}°  RMSE={post_rmse_b:.4f}°')
print(f'Triggers before t=20s:   {baseline["triggers_pre"]}')
print(f'Triggers after t=20s:    {baseline["triggers_post"]}')

## 8. Final evaluation — with damage event at t = 20 s

Same 60 s episode and same trained agent, but now the env has a `DamageProfile` that fires two `section_loss` events at t = 20 s (30 % of `left_tip` and 30 % of `right_tip`). After the event, the env recomputes mass/area/inertia and the longitudinal ODE adds the strip-theory `ΔCy` and `ΔMy` corrections — the plant the agent observes is genuinely different from t = 20 s onward.

Online actor/critic learning stays on. The plant NN is **not** retrained — it remains the offline-fit healthy-aircraft model. So the closed loop has to compensate purely through actor/critic updates whenever the event trigger fires.

In [ ]:
damage_profile = DamageProfile(events=[
    DamageEvent(
        trigger_time=DAMAGE_TIME, event_type='section_loss',
        payload={'section': 'left_tip', 'loss_fraction': 0.30},
        label='left_tip_30pct_loss',
    ),
    DamageEvent(
        trigger_time=DAMAGE_TIME, event_type='section_loss',
        payload={'section': 'right_tip', 'loss_fraction': 0.30},
        label='right_tip_30pct_loss',
    ),
])

def damaged_env():
    return make_env(damage_profile=damage_profile)

damaged = run_episode(agent, damaged_env, learn=True)
n_alpha = len(damaged['alpha'])
ref_deg = np.rad2deg(reference_signals[0, :n_alpha])
pre = np.arange(int(5.0 / DT), DAMAGE_STEP)
post = np.arange(DAMAGE_STEP + int(2.0 / DT), n_alpha)

pre_rmse_d = float(np.sqrt(np.mean((damaged['alpha'][pre] - ref_deg[pre]) ** 2)))
pre_mae_d  = float(np.mean(np.abs(damaged['alpha'][pre] - ref_deg[pre])))
post_rmse_d = float(np.sqrt(np.mean((damaged['alpha'][post] - ref_deg[post]) ** 2)))
post_mae_d  = float(np.mean(np.abs(damaged['alpha'][post] - ref_deg[post])))

print('=== With damage (30% bilateral wing-tip loss at t=20s) ===')
print(f'Pre-damage  (5–20 s):    MAE={pre_mae_d:.4f}°  RMSE={pre_rmse_d:.4f}°')
print(f'Post-damage (22–60 s):   MAE={post_mae_d:.4f}°  RMSE={post_rmse_d:.4f}°')
print(f'Triggers before t=20s:   {damaged["triggers_pre"]}')
print(f'Triggers after t=20s:    {damaged["triggers_post"]}')
if damaged['triggered_events']:
    print('Damage events:')
    for t, lab in damaged['triggered_events']:
        print(f'  t={t:.2f}s : {lab}')

## 9. Side-by-side plots

Three panels: α tracking, residual elevator, pitch rate. The red dashed line marks the damage event at t = 20 s.

In [ ]:
n_plot = min(len(baseline['alpha']), len(damaged['alpha']))
ref_deg = np.rad2deg(reference_signals[0, :n_plot])
tps_plot = tps[:n_plot]

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(tps_plot, ref_deg, 'k--', alpha=0.6, label=r'$\alpha_{\mathrm{ref}}$')
axes[0].plot(tps_plot, baseline['alpha'][:n_plot], alpha=0.7, label='ET-DHP (no damage)')
axes[0].plot(tps_plot, damaged['alpha'][:n_plot], alpha=0.9, label='ET-DHP (with damage @ t=20s)')
axes[0].axvline(DAMAGE_TIME, color='red', linestyle='--', alpha=0.4, label=f'damage @ t={DAMAGE_TIME:.0f}s')
axes[0].set_ylabel(r'$\alpha$, deg')
axes[0].set_title('ET-DHP α tracking on nonlinear F-16 — baseline vs damage')
axes[0].grid(True); axes[0].legend(loc='upper right')

axes[1].plot(tps_plot, baseline['u'][:n_plot], alpha=0.7, label='no damage')
axes[1].plot(tps_plot, damaged['u'][:n_plot], alpha=0.9, label='with damage')
axes[1].axvline(DAMAGE_TIME, color='red', linestyle='--', alpha=0.4)
axes[1].axhline(cfg.u_bound, color='k', linestyle=':', linewidth=0.7)
axes[1].axhline(-cfg.u_bound, color='k', linestyle=':', linewidth=0.7)
axes[1].set_ylabel('residual $\\delta_e$, deg')
axes[1].grid(True); axes[1].legend(loc='upper right')

axes[2].plot(tps_plot, baseline['wz'][:n_plot], alpha=0.7, label='no damage')
axes[2].plot(tps_plot, damaged['wz'][:n_plot], alpha=0.9, label='with damage')
axes[2].axvline(DAMAGE_TIME, color='red', linestyle='--', alpha=0.4)
axes[2].set_ylabel(r'$\omega_z$, deg/s')
axes[2].set_xlabel('time, s')
axes[2].grid(True); axes[2].legend(loc='upper right')

plt.tight_layout(); plt.show()

## 10. Summary

**What the example showed:**

* ET-DHP can run with the new damage subsystem without any modification: pass `damage_profile=...` to the env constructor and the rest of the pipeline (state transform, feedforward, agent loop) is unchanged.
* The Lipschitz event trigger correctly responds to the post-damage tracking error — trigger count jumps after t = 20 s as the actor and critic try to compensate. In the illustrative run logged above, post-damage triggers ≈ 2× the pre-damage rate.
* The closed loop degrades but stays bounded. Pre-damage tracking is dominated by the feedforward; after damage, the actor's `±2°` residual is not always enough to fully cancel the changed plant gain — hence the higher post-damage RMSE.

**Why ET-DHP is at a disadvantage here:**

* The plant network was pre-trained on the **healthy** aircraft. Its Jacobians `F = ∂f/∂x` and `G = ∂f/∂u` no longer reflect the post-damage dynamics, so the closed-form policy `u* = u_b · tanh(-½γ R⁻¹ Gᵀ λ)` uses a stale gain.
* The actor/critic still re-train through the event trigger, but they cannot recover what the plant model is missing.

**Compared with iADP** (`example_iadp_damage_f16.py`):

* iADP identifies the local plant model online via RLS, so the post-damage gain `Ĝ` is picked up within milliseconds and the closed-form iADP policy adapts immediately. Post-damage RMSE ends up close to the no-damage baseline.
* ET-DHP relies on a fixed plant NN, so its adaptation under damage is bounded by what the actor/critic can compensate for.

**Possible extensions:**

* Re-run plant-NN updates (`agent.fit_plant_model(...)`) on a sliding window of recent transitions — effectively making the plant model online too.
* Use the `damage_observable=True` env flag so the agent sees the `DamageState` vector in its observation and can condition the actor on it.
* Increase `u_bound` from 2° to 5° so the residual can compensate larger plant changes.